# Framewise-displacement quality control

This notebook shows the largest percentage of high-motion volumes observed for each subject across their resting-state scans. A volume is classified as high motion when framewise displacement is at least 0.5 mm. This notebook REQUIRES access to the full research research data store of the trial to run properly

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Calculate scan-level motion

In [ ]:
#Path to relevant directory in the RDS
base_dir = #PATH TO RDS DIRECTORY
subjects = [subject for subject in range(1, 51) if subject not in {27, 36, 42}]

rows = []

for subject in subjects:
    confounds_files = sorted(
        (base_dir / f"sub-{subject}").glob(
            "ses-*/func/*_task-rest_run-*_desc-confounds_timeseries.tsv"
        )
    )

    for confounds_file in confounds_files:
        session = int(confounds_file.parents[1].name.split("-")[1])
        run = confounds_file.name.split("_run-")[1].split("_")[0]
        fd = pd.read_csv(
            confounds_file,
            sep="	",
            usecols=["framewise_displacement"],
        )["framewise_displacement"].fillna(0)

        rows.append({
            "subject": subject,
            "session": session,
            "run": run,
            "pct_fd_ge_0_5": 100 * fd.ge(0.5).mean(),
        })

scan_qc = pd.DataFrame(rows)
worst_fd = (
    scan_qc.loc[
        scan_qc.groupby("subject")["pct_fd_ge_0_5"].idxmax()
    ]
    .sort_values("subject")
    .reset_index(drop=True)
)

print("Subjects:", len(worst_fd))

## Worst scan per subject

In [ ]:
values = worst_fd["pct_fd_ge_0_5"].to_numpy()
x_positions = 1 + np.random.default_rng(42).uniform(-0.14, 0.14, len(worst_fd))
above_threshold = values > 15

fig, ax = plt.subplots(figsize=(7, 6))

violin = ax.violinplot(
    values,
    positions=[1],
    widths=0.50,
    showmeans=False,
    showmedians=False,
    showextrema=False,
    bw_method=0.35,
)
violin["bodies"][0].set_facecolor("#BFD8F0")
violin["bodies"][0].set_edgecolor("#4D82AA")
violin["bodies"][0].set_linewidth(1.3)
violin["bodies"][0].set_alpha(0.85)

ax.scatter(
    x_positions[~above_threshold],
    values[~above_threshold],
    s=40,
    facecolor="#BFD8F0",
    edgecolor="#4D82AA",
    linewidth=0.9,
    alpha=0.95,
    zorder=4,
)

ax.scatter(
    x_positions[above_threshold],
    values[above_threshold],
    s=52,
    facecolor="#E8B4B8",
    edgecolor="#A64D57",
    linewidth=1.2,
    alpha=0.98,
    zorder=5,
)

for x, y, subject in zip(
    x_positions[above_threshold],
    values[above_threshold],
    worst_fd.loc[above_threshold, "subject"],
):
    ax.text(
        x + 0.015,
        y + 0.35,
        str(subject),
        fontsize=9.5,
        color="#A64D57",
    )

ax.axhline(
    15,
    color="#777777",
    linestyle=(0, (2, 2)),
    linewidth=1.3,
)
ax.text(
    0.62,
    15.6,
    "15% exclusion threshold",
    fontsize=9.5,
    color="#666666",
)

ax.text(
    0.97,
    0.96,
    f"{above_threshold.sum()}/{len(worst_fd)} subjects above 15%",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10.5,
    color="#555555",
    bbox={
        "boxstyle": "round,pad=0.35",
        "facecolor": "white",
        "edgecolor": "#D0D0D0",
        "linewidth": 0.8,
    },
)

ax.set_xlim(0.55, 1.45)
ax.set_ylim(
    0,
    max(20, np.ceil((max(values.max(), 15) + 5) / 5) * 5),
)
ax.set_xticks([1], [""])
ax.set_xlabel("Subject", fontsize=12)
ax.set_ylabel("Flagged time points (%)", fontsize=12)
ax.grid(axis="y", linewidth=0.7, alpha=0.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.subplots_adjust(
    top=0.88,
    bottom=0.14,
    left=0.14,
    right=0.97,
)

plt.show()